In [2]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [2]:
df = pd.read_csv('online_retail_II.csv', encoding='utf-8')
print(f'строк загружено {len(df)}')
print(f'Колонки: {df.columns.tolist()}')
df.head()

строк загружено 1067371
Колонки: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print("--- Размер датасета ---")
print(df.shape)
print('\n --- Типы данных ---')
print(df.dtypes)
print('\n --- Пропуски по колонкам ---')
print(df.isnull().sum())
print('\n --- Статистика по полям ---')
print(df.describe())

--- Размер датасета ---
(1067371, 8)

 --- Типы данных ---
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

 --- Пропуски по колонкам ---
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

 --- Статистика по полям ---
           Quantity         Price    Customer ID
count  1.067371e+06  1.067371e+06  824364.000000
mean   9.938898e+00  4.649388e+00   15324.638504
std    1.727058e+02  1.235531e+02    1697.464450
min   -8.099500e+04 -5.359436e+04   12346.000000
25%    1.000000e+00  1.250000e+00   13975.000000
50%    3.000000e+00  2.100000e+00   15255.000000
75%    1.000000e+01  4.150000e+00   16797.000000
max    8.099500e+04  3.897000e+04   18287.000000


In [4]:
df_clean = df.copy()

In [5]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Удалено дубликатов {before - len(df_clean)}')

Удалено дубликатов 34335


In [6]:
df_clean = df_clean.dropna(subset=['Customer ID', 'Description'])
print(f'После удаления пропусков {len(df_clean)}')

После удаления пропусков 797885


In [7]:
# --- Возвраты отдельно (Invoice начинается с 'C') ---
returns = df_clean[df_clean['Invoice'].astype('str').str.startswith('C')]
df_clean = df_clean[~df_clean['Invoice'].astype('str').str.startswith('C')]
print(f'Возвратов сохранено отдельно {len(returns)}')

Возвратов сохранено отдельно 18390


In [8]:
# --- Отрицательные и нулевые значения ---
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['Price'] > 0]
print(f'После удаления отпицательных {len(df_clean)}')

После удаления отпицательных 779425


In [9]:
# --- Служебные StockCode --
test_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
print(df_clean['StockCode'].isin(test_codes).value_counts())
df_clean = df_clean[~df_clean['StockCode'].isin(test_codes)]
df_clean


StockCode
False    776872
True       2553
Name: count, dtype: int64


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067365,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [10]:
# --- Типы данных ---
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

In [11]:
# --- Новый столбец ---
df_clean['Revenue'] = df_clean['Price'] * df_clean['Quantity']

print(f'\n Строк после очистки {len(df_clean)}')
print(f'\n Уникальных клиентов {df_clean['Customer ID'].nunique()}')
print(f'\n Период: {df_clean['InvoiceDate'].min()} - {df_clean['InvoiceDate'].max()}')


 Строк после очистки 776872

 Уникальных клиентов 5862

 Период: 2009-12-01 07:45:00 - 2011-12-09 12:50:00


In [4]:
load_dotenv(encoding='utf-8')
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

with engine.connect() as conn:
    print("Соединение успешно")

Соединение успешно


In [15]:
print('Заливаем transactions...')
df_clean.to_sql(
   name='transactions',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=10000
)
print(f'Таблица transactions залита {len(df_clean)} строк')

Заливаем transactions...
Таблица transactions залита 776872 строк


In [16]:
print('Заливаем returns...')
returns.to_sql(
    name='returns',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=10000 
)
print(f'Таблица returns залита {len(returns)} строк')

Заливаем returns...
Таблица returns залита 18390 строк


In [18]:
with engine.connect() as conn:
    results = pd.read_sql(text("""
        SELECT
            COUNT(*) as total_rows,
            COUNT(DISTINCT "Customer ID") as customers,
            COUNT(DISTINCT "StockCode") as products,
            MIN("InvoiceDate") as date_from,
            MAX("InvoiceDate") as date_to,
            ROUND(SUM("Revenue")::numeric, 2) as total_revenue
        FROM transactions
    """), conn)

    print(results)

   total_rows  customers  products           date_from             date_to  \
0      776872       5862      4625 2009-12-01 07:45:00 2011-12-09 12:50:00   

   total_revenue  
0    17085624.29  
